<a href="https://colab.research.google.com/github/DezinTI/Data-Science-com-Pandas/blob/main/Trabalho_DataScience_Finalizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho de Data Science
**Autores:** André Augustinho da Costa (RGM: 39483380), Ricardo Augusto Matias da Luz (RGM: 37701568)

## 1. Introdução
Neste trabalho vamos aprender:
- Carregar qualquer base binária (.csv)
- Criar/ajustar a coluna target para classificação binária
- Verificar balanceamento
- Treinar dois modelos (Regressão Logística & Random Forest)
- Avaliar métricas, curvas e histogramas
- Explorar pontos de corte para automação/manual

## 2. Importando as Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, classification_report, auc
)

sns.set(style="whitegrid")

## 3. Carregando a Base de Dados
Faça upload do seu arquivo CSV na célula abaixo. Se não existir uma coluna target (classe), veja no print quais colunas existem e crie a sua.

In [ ]:
from google.colab import files
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Colunas disponíveis:", df.columns.tolist())
print(df.head())

## 4. Verificando o Balanceamento da Base

In [ ]:
print(df['target'].value_counts(normalize=True)*100)
sns.countplot(data=df, x='target', palette="Set2")
plt.title("Distribuição da variável alvo (target)")
plt.show()

## 5. Pré-processamento dos Dados

In [ ]:
col_target = 'target'
col_text = None
for col in df.columns:
    if df[col].dtype == "object" and col != col_target:
        if df[col].astype(str).str.len().mean() > 15:
            col_text = col
            break

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
if col_text and col_text in categorical_cols:
    categorical_cols.remove(col_text)
if col_target in numerical_cols:
    numerical_cols.remove(col_target)
elif col_target in categorical_cols:
    categorical_cols.remove(col_target)

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='mean'), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

df_features = df.drop(columns=[col_target])
if col_text:
    vectorizer = TfidfVectorizer()
    X_text = vectorizer.fit_transform(df_features[col_text].astype(str))
    df_no_text = df_features.drop(columns=[col_text])
    X_tab = preprocessor.fit_transform(df_no_text)
    from scipy.sparse import hstack
    X = hstack([X_tab, X_text])
else:
    X = preprocessor.fit_transform(df_features)

y = df[col_target]

try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
except Exception as e:
    print(f"Aviso: {e}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Treino:", X_train.shape, "Teste:", X_test.shape)

## 6. Treinando e Avaliando os Modelos

In [ ]:
lr = LogisticRegression(max_iter=500)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("=== Regressão Logística ===")
print(classification_report(y_test, y_pred_lr))
print("Acurácia:", accuracy_score(y_test, y_pred_lr))

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))
print("Acurácia:", accuracy_score(y_test, y_pred_rf))

## 7. GridSearchCV (Otimização de Hiperparâmetros)

In [ ]:
unique_train = np.unique(y_train, return_counts=True)[1]
if np.min(unique_train) >= 2:
    param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    }
    grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=2, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    print("Melhores parâmetros:", grid.best_params_)
    melhor_rf = grid.best_estimator_
    y_pred_grid = melhor_rf.predict(X_test)
    print(classification_report(y_test, y_pred_grid))
else:
    print("GridSearchCV não rodou: poucas amostras em alguma classe.")
    melhor_rf = rf

## 8. Avaliação Avançada: Métricas e Curvas

In [ ]:
modelo = melhor_rf
y_prob = modelo.predict_proba(X_test)[:, 1]
print("Acurácia:", accuracy_score(y_test, modelo.predict(X_test)))
print("Precisão:", precision_score(y_test, modelo.predict(X_test)))
print("Recall:", recall_score(y_test, modelo.predict(X_test)))
print("F1-score:", f1_score(y_test, modelo.predict(X_test)))
print("AUC-ROC:", roc_auc_score(y_test, y_prob))
print("AUC-PR:", average_precision_score(y_test, y_prob))

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC')
plt.legend(loc='lower right')
plt.show()

precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall, precision)
plt.figure()
plt.plot(recall, precision, label=f'PR curve (area = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precisão-Recall')
plt.legend(loc='lower left')
plt.show()

## 9. Função para Avaliação dos Pontos de Corte

In [ ]:
def avaliar_cortes(y_true, y_prob, corte_neg=0.3, corte_pos=0.7):
    y_pred = []
    for p in y_prob:
        if p <= corte_neg:
            y_pred.append(0)
        elif p >= corte_pos:
            y_pred.append(1)
        else:
            y_pred.append(-1)
    df_cortes = pd.DataFrame({"Prob": y_prob, "Target": y_true, "Pred": y_pred})
    df_validos = df_cortes[df_cortes["Pred"] != -1]
    if len(df_validos) > 0:
        acc = accuracy_score(df_validos["Target"], df_validos["Pred"])
        prec = precision_score(df_validos["Target"], df_validos["Pred"], zero_division=0)
        rec = recall_score(df_validos["Target"], df_validos["Pred"], zero_division=0)
        f1 = f1_score(df_validos["Target"], df_validos["Pred"], zero_division=0)
    else:
        acc = prec = rec = f1 = 0
    total = len(df_cortes)
    pct_automacao_neg = (df_cortes['Pred'] == 0).sum() / total * 100
    pct_automacao_pos = (df_cortes['Pred'] == 1).sum() / total * 100
    pct_analise_manual = (df_cortes['Pred'] == -1).sum() / total * 100
    print(f"--- Avaliação com cortes neg={corte_neg}, pos={corte_pos} ---")
    print("Tamanho total:", len(df_cortes))
    print("Casos manuais:", sum(df_cortes['Pred'] == -1))
    print(f"Accuracy={acc:.3f}, Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}")
    print(f"Automático negativo: {pct_automacao_neg:.2f}% | Automático positivo: {pct_automacao_pos:.2f}% | Manual: {pct_analise_manual:.2f}%\n")
    return df_cortes

## 10. Testando vários pontos de corte

In [ ]:
modelo_escolhido = melhor_rf
y_prob = modelo_escolhido.predict_proba(X_test)[:, 1]
y_true = y_test.values

for corte_neg in [0.3, 0.4, 0.5]:
    for corte_pos in [0.7, 0.8, 0.9]:
        avaliar_cortes(y_true, y_prob, corte_neg, corte_pos)
        print('-' * 50)

## 11. Gráfico dos Pontos de Corte (População x Probabilidade)

In [ ]:
step = 0.1  # você pode mudar para outro valor (ex: 0.05)
bins = np.arange(0, 1 + step, step)
df_plot = pd.DataFrame({'prob_pos': y_prob, 'target': y_true})
df_plot['bin'] = pd.cut(df_plot['prob_pos'], bins=bins, include_lowest=True)
pop_total = df_plot['bin'].value_counts(normalize=True).sort_index() * 100
pop_pos = df_plot[df_plot['target'] == 1]['bin'].value_counts(normalize=True).sort_index() * 100
labels = pop_total.index.astype(str)
x = np.arange(len(labels))
width = 0.4

plt.figure(figsize=(12, 6))
bars1 = plt.bar(x - width/2, pop_total.values, width=width, color='blue', label='População Geral')
bars2 = plt.bar(x + width/2, pop_pos.values, width=width, color='red', label='Target Positivo')

for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.2, f'{height:.1f}%', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.2, f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

plt.xticks(x, labels, rotation=45)
plt.xlabel('Faixa de Probabilidade (%)')
plt.ylabel('Percentual Populacional (%)')
plt.title('Distribuição de Probabilidades - Barras Lado a Lado')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()